# Module 02, Unit 04 — The Multivariable Chain Rule

> **Threat Surfaces: Multivariable Calculus for AI Security**
> fischer³ Education | Module 02 | Unit 04

| | |
|---|---|
| **Estimated time** | 55–65 min |
| **Exercises** | Download PDF from the course repository |
| **Statistical bridge** | Backpropagation as the chain rule; gradient of composed statistical models [GLM] |
| **Refresher** | None required |

---

## Learning Objectives

By the end of this unit you will be able to:

- [ ] State the multivariable chain rule for compositions $f \circ \boldsymbol{g}$ and express it as a matrix-vector product
- [ ] Define the Jacobian matrix of a vector-valued function and compute it for low-dimensional examples
- [ ] Apply the chain rule to compute gradients of composed functions — polynomials, exponentials, and activations composed with linear maps
- [ ] Trace the forward and backward pass of a simple computational graph, identifying where the chain rule is applied at each node
- [ ] Explain why backpropagation is algorithmically equivalent to the multivariable chain rule applied in reverse topological order
- [ ] Compute the gradient of the logistic regression log-likelihood with respect to the weights using the chain rule

---

## Why This Unit

The multivariable chain rule is the mathematical engine behind every gradient computation in modern machine learning. When you call `.backward()` in PyTorch or JAX, the framework is applying the chain rule — recursively, automatically, efficiently — through the computational graph of your model. This unit builds the mathematical foundation that makes that process intelligible rather than magical.

We approach the chain rule in three layers of increasing sophistication:
1. The scalar chain rule from Module 00-03, generalised to multiple inputs and outputs.
2. The Jacobian chain rule — derivatives of vector-valued functions composed with vector-valued functions.
3. Backpropagation — the reverse-mode algorithm that evaluates the chain rule efficiently through a computational graph.

The statistical bridge is logistic regression: a composed function whose gradient, derived via the chain rule, is the foundation of gradient-based GLM training.

---

## 1. The Chain Rule: Scalar Compositions

### 1.1 One Intermediate Variable

If $z = f(y)$ and $y = g(x_1, x_2)$, then $z = f(g(x_1, x_2))$ and:

$$
\frac{\partial z}{\partial x_i} = \frac{dz}{dy} \cdot \frac{\partial y}{\partial x_i}.
$$

The single-variable chain rule (Module 00-03) generalises by multiplying the outer derivative by the partial of the inner function with respect to each input variable.

**Example.** Let $z = e^{x_1^2 + x_2^2}$. Setting $y = x_1^2 + x_2^2$:

$$
\frac{\partial z}{\partial x_1} = e^y \cdot 2x_1 = 2x_1 e^{x_1^2+x_2^2}, \qquad \frac{\partial z}{\partial x_2} = 2x_2 e^{x_1^2+x_2^2}.
$$

### 1.2 Multiple Intermediate Variables

If $z = f(y_1, y_2, \ldots, y_m)$ and each $y_k = g_k(x_1, \ldots, x_n)$, then:

$$
\frac{\partial z}{\partial x_i} = \sum_{k=1}^{m} \frac{\partial z}{\partial y_k} \cdot \frac{\partial y_k}{\partial x_i}.
$$

Each path from $x_i$ to $z$ through an intermediate variable $y_k$ contributes one term to the sum. This **sum-over-paths** structure is the core of the chain rule and the core of backpropagation.

**Example.** Let $z = y_1^2 + y_1 y_2$, where $y_1 = x_1 + x_2$ and $y_2 = x_1 x_2$.

$$
\frac{\partial z}{\partial x_1} = \frac{\partial z}{\partial y_1}\frac{\partial y_1}{\partial x_1} + \frac{\partial z}{\partial y_2}\frac{\partial y_2}{\partial x_1}
= (2y_1 + y_2)(1) + (y_1)(x_2)
= 2(x_1+x_2) + x_1 x_2 + (x_1+x_2)x_2.
$$

---

## 2. The Jacobian and the Matrix Chain Rule

### 2.1 The Jacobian Matrix

For a vector-valued function $\boldsymbol{g}: \mathbb{R}^n \to \mathbb{R}^m$, the **Jacobian** at $\boldsymbol{x}$ is the $m \times n$ matrix of all first-order partial derivatives:

$$
\boldsymbol{J}_g(\boldsymbol{x}) = \begin{pmatrix}
\frac{\partial g_1}{\partial x_1} & \cdots & \frac{\partial g_1}{\partial x_n} \\
\vdots & \ddots & \vdots \\
\frac{\partial g_m}{\partial x_1} & \cdots & \frac{\partial g_m}{\partial x_n}
\end{pmatrix}.
$$

The $i$-th row is the gradient of the $i$-th output function $g_i$. The $j$-th column collects all partial derivatives with respect to input $x_j$.

When $m = 1$ (scalar output), the Jacobian is a $1 \times n$ row vector — the transpose of the gradient: $\boldsymbol{J}_f = \nabla f^{\top}$.

**Example.** Let $\boldsymbol{g}(x,y) = (x^2 + y,\; e^{xy})^{\top}$.
$$
\boldsymbol{J}_g = \begin{pmatrix} 2x & 1 \\ ye^{xy} & xe^{xy} \end{pmatrix}.
$$

### 2.2 The Chain Rule as Matrix Multiplication

If $\boldsymbol{h} = f \circ \boldsymbol{g}$, i.e. $\boldsymbol{h}(\boldsymbol{x}) = f(\boldsymbol{g}(\boldsymbol{x}))$ where $\boldsymbol{g}: \mathbb{R}^n \to \mathbb{R}^m$ and $f: \mathbb{R}^m \to \mathbb{R}^p$, then:

$$
\boldsymbol{J}_h(\boldsymbol{x}) = \boldsymbol{J}_f(\boldsymbol{g}(\boldsymbol{x})) \cdot \boldsymbol{J}_g(\boldsymbol{x}).
$$

The Jacobian of the composition is the product of the Jacobians: $(p \times m) \cdot (m \times n) = (p \times n)$.

For the special case where the outer function is scalar ($p = 1$), $f: \mathbb{R}^m \to \mathbb{R}$, the chain rule gives the gradient:

$$
\nabla_x (f \circ \boldsymbol{g}) = \boldsymbol{J}_g^{\top} \cdot \nabla_y f,
$$

where $\nabla_y f$ is the gradient of $f$ with respect to its input $\boldsymbol{y} = \boldsymbol{g}(\boldsymbol{x})$. This is the formula used in backpropagation at each layer.

---

## 3. Computational Graphs and Backpropagation

### 3.1 What Is a Computational Graph?

A **computational graph** is a directed acyclic graph (DAG) in which:
- **Leaf nodes** are the inputs (parameters $\boldsymbol{\theta}$ and data $\boldsymbol{x}$).
- **Intermediate nodes** are the results of elementary operations: addition, multiplication, exponentiation, etc.
- **The root node** is the final output (the loss $\mathcal{L}$).

Every differentiable function can be decomposed into a computational graph of elementary operations, each of which has a known derivative.

### 3.2 Forward Pass

In the **forward pass**, we evaluate the function by traversing the graph from leaves to root, computing each intermediate value in topological order. At the end of the forward pass, we have the numerical value of the loss and the values of all intermediate nodes.

### 3.3 Backward Pass (Backpropagation)

In the **backward pass**, we compute the gradient of the loss with respect to every parameter. We traverse the graph in **reverse topological order** (from root to leaves), applying the chain rule at each node.

At each node $v$ in the graph, we accumulate the **upstream gradient** (the partial derivative of the loss with respect to the output of $v$) and multiply it by the **local Jacobian** (the derivative of $v$'s output with respect to its inputs) to produce the gradient flowing into the inputs of $v$.

Formally, if node $v$ computes $y = h(x_1, \ldots, x_k)$ and the upstream gradient is $\frac{\partial \mathcal{L}}{\partial y}$, then the gradient flowing into input $x_j$ is:

$$
\frac{\partial \mathcal{L}}{\partial x_j} = \frac{\partial \mathcal{L}}{\partial y} \cdot \frac{\partial y}{\partial x_j}.
$$

This is the chain rule. Backpropagation is nothing more than the systematic, efficient application of the chain rule to a computational graph.

### 3.4 Why Reverse Mode Is Efficient

There are two ways to apply the chain rule through a graph: **forward mode** (computing Jacobian-vector products, propagating from inputs to outputs) and **reverse mode** (computing vector-Jacobian products, propagating from outputs to inputs).

For a scalar loss $\mathcal{L}: \mathbb{R}^p \to \mathbb{R}$ with $p$ parameters:
- **Forward mode** requires $p$ passes (one per parameter) to compute the full gradient.
- **Reverse mode (backprop)** requires only **one pass** to compute the gradient with respect to *all* $p$ parameters simultaneously.

This is why backpropagation scales to millions of parameters while forward-mode differentiation would not. The mathematical reason: the gradient $\nabla_{\boldsymbol{\theta}}\mathcal{L}$ is a vector of $p$ numbers; reverse mode computes all $p$ in one sweep because it starts from the scalar output and accumulates gradients backward.

### 3.5 A Worked Computational Graph Example

**Function:** $\mathcal{L}(w_1, w_2) = \left(\sigma(w_1 x + w_2) - y\right)^2$ where $\sigma(t) = 1/(1+e^{-t})$ is the sigmoid, and $x, y$ are fixed data.

**Decomposition into elementary operations:**
$$
a = w_1 x + w_2, \quad b = e^{-a}, \quad c = 1 + b, \quad \hat{y} = 1/c, \quad d = \hat{y} - y, \quad \mathcal{L} = d^2.
$$

**Forward pass:** evaluate $a, b, c, \hat{y}, d, \mathcal{L}$ in order.

**Backward pass** (chain rule, from $\mathcal{L}$ to $w_1, w_2$):

| Node | Local derivative | Upstream | Downstream |
|---|---|---|---|
| $\mathcal{L} = d^2$ | $\frac{d\mathcal{L}}{dd} = 2d$ | 1 (root) | $\frac{\partial\mathcal{L}}{\partial d} = 2d$ |
| $d = \hat{y} - y$ | $\frac{\partial d}{\partial \hat{y}} = 1$ | $2d$ | $\frac{\partial\mathcal{L}}{\partial \hat{y}} = 2d$ |
| $\hat{y} = 1/c$ | $\frac{\partial\hat{y}}{\partial c} = -1/c^2$ | $2d$ | $\frac{\partial\mathcal{L}}{\partial c} = -2d/c^2$ |
| $c = 1+b$ | $\frac{\partial c}{\partial b} = 1$ | above | $\frac{\partial\mathcal{L}}{\partial b} = -2d/c^2$ |
| $b = e^{-a}$ | $\frac{\partial b}{\partial a} = -e^{-a} = -b$ | above | $\frac{\partial\mathcal{L}}{\partial a} = 2db/c^2$ |
| $a = w_1 x + w_2$ | $\frac{\partial a}{\partial w_1} = x,\;\frac{\partial a}{\partial w_2} = 1$ | above | $\frac{\partial\mathcal{L}}{\partial w_1} = 2dbx/c^2,\;\frac{\partial\mathcal{L}}{\partial w_2} = 2db/c^2$ |

With $\hat{y} = \sigma(a)$ and $d = \hat{y} - y$, simplification gives the elegant result:

$$
\frac{\partial\mathcal{L}}{\partial w_1} = 2(\hat{y} - y)\,\hat{y}(1-\hat{y})\,x, \qquad \frac{\partial\mathcal{L}}{\partial w_2} = 2(\hat{y}-y)\,\hat{y}(1-\hat{y}).
$$

Note that $\hat{y}(1-\hat{y}) = \sigma(a)(1-\sigma(a)) = \sigma'(a)$ — the derivative of the sigmoid — confirming the chain rule result.

---

## 4. Statistical Bridge — Logistic Regression Gradient via Chain Rule [GLM]

### 4.1 The Model

Logistic regression models binary outcomes $y_i \in \{0, 1\}$ as Bernoulli$(\hat{p}_i)$, where

$$
\hat{p}_i = \sigma(\boldsymbol{x}_i^{\top}\boldsymbol{w}) = \frac{1}{1 + e^{-\boldsymbol{x}_i^{\top}\boldsymbol{w}}}.
$$

Here $\boldsymbol{x}_i \in \mathbb{R}^p$ is the $i$-th feature vector and $\boldsymbol{w} \in \mathbb{R}^p$ is the weight vector.

### 4.2 The Log-Likelihood and Loss

The log-likelihood is

$$
\ell(\boldsymbol{w}) = \sum_{i=1}^{n} \left[ y_i \log \hat{p}_i + (1-y_i)\log(1-\hat{p}_i) \right].
$$

The negative log-likelihood — the **binary cross-entropy loss** used in gradient descent — is $\mathcal{L}(\boldsymbol{w}) = -\ell(\boldsymbol{w})$.

### 4.3 Gradient via the Chain Rule

We compute $\nabla_{\boldsymbol{w}} \ell$ using the chain rule. Write $a_i = \boldsymbol{x}_i^{\top}\boldsymbol{w}$ (the linear predictor). Then $\hat{p}_i = \sigma(a_i)$.

**Step 1 — derivative of $\ell$ with respect to $a_i$:**

$$
\frac{\partial \ell}{\partial a_i} = \frac{\partial}{\partial a_i}\left[y_i \log\sigma(a_i) + (1-y_i)\log(1-\sigma(a_i))\right].
$$

Using $\frac{d}{da}\log\sigma(a) = 1 - \sigma(a)$ and $\frac{d}{da}\log(1-\sigma(a)) = -\sigma(a)$:

$$
\frac{\partial \ell}{\partial a_i} = y_i(1-\hat{p}_i) - (1-y_i)\hat{p}_i = y_i - \hat{p}_i.
$$

**Step 2 — chain rule to $\boldsymbol{w}$:**

Since $a_i = \boldsymbol{x}_i^{\top}\boldsymbol{w}$, we have $\frac{\partial a_i}{\partial \boldsymbol{w}} = \boldsymbol{x}_i$. By the chain rule:

$$
\frac{\partial \ell}{\partial \boldsymbol{w}} = \sum_{i=1}^{n} \frac{\partial \ell}{\partial a_i} \cdot \frac{\partial a_i}{\partial \boldsymbol{w}} = \sum_{i=1}^{n} (y_i - \hat{p}_i)\,\boldsymbol{x}_i.
$$

**Matrix form:** stacking observations into $\boldsymbol{X} \in \mathbb{R}^{n \times p}$ and $\boldsymbol{y} \in \mathbb{R}^n$:

$$
\boxed{\nabla_{\boldsymbol{w}}\ell = \boldsymbol{X}^{\top}(\boldsymbol{y} - \hat{\boldsymbol{p}}).}
$$

This is the **score of the logistic regression model** — the gradient of the log-likelihood. It is the difference between observed outcomes and predicted probabilities, weighted by the feature matrix. The gradient is zero when $\hat{\boldsymbol{p}} = \boldsymbol{y}$ — i.e., when the model predictions equal the data, which only happens in the limit of perfect fit.

The gradient descent update rule for logistic regression is:

$$
\boldsymbol{w}_{t+1} = \boldsymbol{w}_t + \eta\,\boldsymbol{X}^{\top}(\boldsymbol{y} - \hat{\boldsymbol{p}}_t),
$$

where $\hat{\boldsymbol{p}}_t = \sigma(\boldsymbol{X}\boldsymbol{w}_t)$ are the current predictions. This is the **logistic regression update** — derived entirely from the chain rule.

---

## Python: Visualization & Verification

1. **Symbolic chain rule** — verify Jacobian computations with SymPy
2. **Computational graph trace** — implement the forward and backward pass of a simple computation by hand, verifying against numerical differentiation
3. **Statistical bridge** — implement logistic regression gradient ascent from scratch using the chain rule gradient formula

In [ ]:
import subprocess, sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

for pkg in ["numpy", "matplotlib", "sympy", "scipy"]:
    install(pkg)

print("All packages installed.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6), 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'lines.linewidth': 2, 'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

print('Setup complete.')

---

## Section 1 — Symbolic Chain Rule & Jacobians

In [ ]:
x1, x2, w1, w2, t = sp.symbols('x1 x2 w1 w2 t', real=True)

# --- Jacobian of g: R^2 -> R^2 ---
g1 = x1**2 + x2
g2 = sp.exp(x1 * x2)
g  = sp.Matrix([g1, g2])
x_vec = sp.Matrix([x1, x2])
J_g   = g.jacobian(x_vec)
print('g(x1, x2) =', g.T)
print('Jacobian J_g =')
sp.pprint(J_g)

# --- Composition: f(y1,y2) = y1^2 * y2 composed with g ---
y1, y2 = sp.symbols('y1 y2', real=True)
f_sym  = y1**2 * y2
# Gradient of f w.r.t. y
grad_f = sp.Matrix([sp.diff(f_sym, y1), sp.diff(f_sym, y2)])
print('\ngrad_f(y) =', grad_f.T)

# Chain rule: grad_x (f o g) = J_g^T * grad_f(g(x))
# Substitute y1=g1, y2=g2 into grad_f
grad_f_at_g = grad_f.subs({y1: g1, y2: g2})
grad_x_fog  = J_g.T * grad_f_at_g
print('\nChain rule: grad_x(f o g) =')
for i, expr in enumerate(grad_x_fog):
    print(f'  Component {i+1}:', sp.simplify(expr))

# Verify by direct differentiation
fog_direct = f_sym.subs({y1: g1, y2: g2})
direct_grad = sp.Matrix([sp.diff(fog_direct, x1), sp.diff(fog_direct, x2)])
print('\nDirect differentiation agrees with chain rule:')
for i in range(2):
    diff_check = sp.simplify(grad_x_fog[i] - direct_grad[i])
    print(f'  Component {i+1} difference:', diff_check, '  (should be 0)')

---

## Section 2 — Manual Backpropagation Through a Computational Graph

We implement the forward and backward pass for the function $\mathcal{L}(w_1, w_2) = (\sigma(w_1 x + w_2) - y)^2$ by hand, then verify against numerical gradient checking.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# Fixed data point
x_data = 2.5
y_data = 1.0

# Parameters at which to evaluate
w1_val = 0.3
w2_val = -0.5

# ============================================================
# FORWARD PASS
# ============================================================
a     = w1_val * x_data + w2_val      # linear combination
b     = np.exp(-a)                     # exp(-a)
c     = 1.0 + b                        # 1 + exp(-a)
y_hat = 1.0 / c                        # sigma(a)
d     = y_hat - y_data                 # residual
L     = d**2                           # squared loss

print('=== Forward Pass ===')
print(f'a     = w1*x + w2 = {a:.4f}')
print(f'b     = exp(-a)   = {b:.4f}')
print(f'c     = 1 + b     = {c:.4f}')
print(f'y_hat = 1/c       = {y_hat:.4f}  (sigmoid)')
print(f'd     = y_hat - y  = {d:.4f}')
print(f'L     = d^2       = {L:.4f}')

# ============================================================
# BACKWARD PASS  (chain rule, reverse order)
# ============================================================
dL_dd    = 2 * d                       # dL/dd
dL_dyhat = dL_dd * 1.0                 # dd/dy_hat = 1
dL_dc    = dL_dyhat * (-1.0 / c**2)   # dy_hat/dc = -1/c^2
dL_db    = dL_dc * 1.0                 # dc/db = 1
dL_da    = dL_db * (-b)               # db/da = -exp(-a) = -b
dL_dw1   = dL_da * x_data             # da/dw1 = x
dL_dw2   = dL_da * 1.0                # da/dw2 = 1

print('\n=== Backward Pass ===')
print(f'dL/dd    = {dL_dd:.4f}')
print(f'dL/dy_hat= {dL_dyhat:.4f}')
print(f'dL/dc    = {dL_dc:.4f}')
print(f'dL/db    = {dL_db:.4f}')
print(f'dL/da    = {dL_da:.4f}')
print(f'dL/dw1   = {dL_dw1:.4f}')
print(f'dL/dw2   = {dL_dw2:.4f}')

# ============================================================
# NUMERICAL GRADIENT CHECK
# ============================================================
eps = 1e-5
def L_func(w1, w2):
    return (sigmoid(w1*x_data + w2) - y_data)**2

num_dw1 = (L_func(w1_val + eps, w2_val) - L_func(w1_val - eps, w2_val)) / (2*eps)
num_dw2 = (L_func(w1_val, w2_val + eps) - L_func(w1_val, w2_val - eps)) / (2*eps)

print('\n=== Numerical Gradient Check ===')
print(f'dL/dw1: analytical = {dL_dw1:.6f},  numerical = {num_dw1:.6f},  diff = {abs(dL_dw1-num_dw1):.2e}')
print(f'dL/dw2: analytical = {dL_dw2:.6f},  numerical = {num_dw2:.6f},  diff = {abs(dL_dw2-num_dw2):.2e}')

### What Do You See?

1. The analytical (backprop) and numerical gradients match to high precision — this is the standard **gradient check** used in practice to debug implementations. The discrepancy should be on the order of $10^{-10}$ or smaller.

2. Each backward step multiplied the upstream gradient by one local derivative — the chain rule applied once per edge of the computational graph. The intermediate values ($a, b, c, \hat{y}, d$) saved in the forward pass are reused in the backward pass. This is why modern frameworks save activations during the forward pass: they are needed to evaluate the local derivatives.

3. The formula for $\frac{\partial \mathcal{L}}{\partial a}$ simplifies to $dL_{da} = 2d \cdot (-b/c^2) = 2(\hat{y}-y)\cdot\hat{y}(1-\hat{y})$. Verify this algebraically using $b/c^2 = e^{-a}/(1+e^{-a})^2 = \sigma(a)(1-\sigma(a))$.

---

## Section 3 — Statistical Bridge: Logistic Regression Gradient Ascent [GLM]

In [ ]:
# ---- Generate synthetic binary classification data ----
rng = np.random.default_rng(seed=42)
n_obs, p_feat = 200, 2

X = rng.standard_normal((n_obs, p_feat))
# Add intercept column
X = np.hstack([X, np.ones((n_obs, 1))])  # shape: (200, 3)
p_feat_aug = 3

# True weights
w_true = np.array([1.5, -2.0, 0.5])
p_true = sigmoid(X @ w_true)
y = rng.binomial(1, p_true).astype(float)

print(f'Data: n={n_obs} observations, p={p_feat} features + intercept')
print(f'True weights: {w_true}')
print(f'Class balance: {y.mean():.3f} positive')

In [ ]:
# ---- Logistic regression by gradient ascent ----

def log_likelihood(w, X, y):
    p_hat = sigmoid(X @ w)
    # Clip to avoid log(0)
    p_hat = np.clip(p_hat, 1e-12, 1-1e-12)
    return np.sum(y * np.log(p_hat) + (1-y) * np.log(1-p_hat))

def gradient_log_lik(w, X, y):
    """Chain rule gradient: X^T (y - p_hat)"""
    p_hat = sigmoid(X @ w)
    return X.T @ (y - p_hat)

# Initialise at zero
w_init = np.zeros(p_feat_aug)
eta    = 0.05     # learning rate
n_iter = 300

w_curr   = w_init.copy()
ell_hist = [log_likelihood(w_curr, X, y)]
w_hist   = [w_curr.copy()]

for _ in range(n_iter):
    grad    = gradient_log_lik(w_curr, X, y)
    w_curr  = w_curr + eta * grad
    ell_hist.append(log_likelihood(w_curr, X, y))
    w_hist.append(w_curr.copy())

w_hist = np.array(w_hist)

print(f'Initial log-likelihood: {ell_hist[0]:.4f}')
print(f'Final   log-likelihood: {ell_hist[-1]:.4f}')
print(f'Final weights: {w_curr.round(4)}')
print(f'True  weights: {w_true}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ---- Left: log-likelihood convergence ----
ax = axes[0]
ax.plot(ell_hist, color=TS_BLUE, lw=2)
ax.set_xlabel('Iteration'); ax.set_ylabel('Log-likelihood')
ax.set_title('Log-likelihood during gradient ascent',
             fontweight='bold', color=TS_GRAY)
ax.axhline(ell_hist[-1], color=TS_AMBER, lw=1, ls='--',
           label=f'Final: {ell_hist[-1]:.2f}')
ax.legend(fontsize=9)

# ---- Middle: weight convergence ----
ax = axes[1]
labels = ['$w_1$', '$w_2$', '$w_3$ (bias)']
colors_w = [TS_BLUE, TS_AMBER, TS_GREEN]
for k, (label, col) in enumerate(zip(labels, colors_w)):
    ax.plot(w_hist[:, k], color=col, lw=2, label=label)
    ax.axhline(w_true[k], color=col, lw=1.2, ls='--', alpha=0.6)
ax.set_xlabel('Iteration'); ax.set_ylabel('Weight value')
ax.set_title('Weight convergence\n(dashed = true value)',
             fontweight='bold', color=TS_GRAY)
ax.legend(fontsize=9)

# ---- Right: decision boundary ----
ax = axes[2]
# Plot data points
ax.scatter(X[y==0, 0], X[y==0, 1], color=TS_RED,  alpha=0.5, s=25, label='$y=0$')
ax.scatter(X[y==1, 0], X[y==1, 1], color=TS_BLUE, alpha=0.5, s=25, label='$y=1$')
# Decision boundary: w1*x1 + w2*x2 + w3 = 0 => x2 = -(w1*x1 + w3)/w2
x1_range = np.linspace(X[:,0].min(), X[:,0].max(), 200)
x2_boundary = -(w_curr[0]*x1_range + w_curr[2]) / w_curr[1]
x2_true_bd  = -(w_true[0]*x1_range + w_true[2]) / w_true[1]
ax.plot(x1_range, x2_boundary, color=TS_AMBER, lw=2,
        label='Estimated boundary')
ax.plot(x1_range, x2_true_bd,  color=TS_GRAY, lw=1.5, ls='--',
        label='True boundary')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Decision boundary (final weights)',
             fontweight='bold', color=TS_GRAY)
ax.legend(fontsize=9); ax.set_ylim(-4, 4)

plt.suptitle('Module 02, Unit 04 — Logistic Regression via Chain Rule Gradient [GLM]',
             fontsize=13, fontweight='bold', color=TS_GRAY)
plt.tight_layout()
plt.savefig('module-02-unit-04-logreg.png', dpi=150, bbox_inches='tight')
plt.show()

### What Do You See?

1. **Convergence** (left): The log-likelihood increases monotonically and plateaus near the MLE. The convergence is smooth because the logistic regression log-likelihood is globally concave (its Hessian is always negative semi-definite).

2. **Weight convergence** (middle): Each weight converges toward its true value (dashed lines). Do all three converge at the same rate? Weights with larger $|x_j|$ entries tend to have larger gradient components and can converge faster — or overshoot with a large learning rate.

3. **Decision boundary** (right): The estimated boundary (orange) closely matches the true boundary (grey dashed), confirming the gradient correctly navigated the log-likelihood surface. The boundary is linear — a consequence of the linear predictor $\boldsymbol{x}^{\top}\boldsymbol{w}$ in logistic regression.

4. **The chain rule connection**: The gradient formula $\boldsymbol{X}^{\top}(\boldsymbol{y} - \hat{\boldsymbol{p}})$ was derived by applying the chain rule twice: once through the sigmoid, once through the linear predictor. Every call to `gradient_log_lik` in this loop is a single backward pass through the computational graph of logistic regression — exactly what an automatic differentiation framework would compute.

In [ ]:
# ============================================================
# Extension Challenge (Optional)
# ============================================================
#
# Add L2 regularisation to the logistic regression objective
# and re-derive the gradient via the chain rule.
#
# Regularised objective:
#   J(w) = ell(w) - (lambda/2) * ||w||^2
#
# Tasks:
#   1. Derive the gradient of J(w) w.r.t. w using the chain rule.
#      The regularisation term contributes -lambda*w to the gradient.
#      Show this step-by-step.
#   2. Implement regularised gradient ascent and compare the
#      final weights for lambda = 0 (unregularised), 0.1, 1.0, 10.0.
#   3. What happens to the decision boundary as lambda increases?
#      Why does regularisation shrink weights toward zero (compare
#      to the Ridge regression geometry from Module 00, Unit 01)?
#   4. At what lambda does the regularised MLE become noticeably
#      biased relative to the true weights?
#
# Your code here:


---

## Summary

| Concept | Key result |
|---|---|
| Chain rule (one intermediate) | $\frac{\partial z}{\partial x_i} = \frac{dz}{dy}\cdot\frac{\partial y}{\partial x_i}$ |
| Chain rule (multiple intermediates) | $\frac{\partial z}{\partial x_i} = \sum_k \frac{\partial z}{\partial y_k}\cdot\frac{\partial y_k}{\partial x_i}$ — sum over all paths |
| Jacobian $\boldsymbol{J}_g$ | $m\times n$ matrix of partials for $\boldsymbol{g}: \mathbb{R}^n \to \mathbb{R}^m$ |
| Jacobian chain rule | $\boldsymbol{J}_{f\circ g} = \boldsymbol{J}_f(\boldsymbol{g}(\boldsymbol{x})) \cdot \boldsymbol{J}_g(\boldsymbol{x})$ |
| Scalar output chain rule | $\nabla_x(f\circ\boldsymbol{g}) = \boldsymbol{J}_g^\top\,\nabla_y f$ |
| Backpropagation | Reverse-mode automatic differentiation — chain rule applied in reverse topological order; one pass computes full gradient |
| Gradient check | Verify analytical gradient against numerical $\frac{f(x+\varepsilon)-f(x-\varepsilon)}{2\varepsilon}$ |
| Logistic regression gradient | $\nabla_w\ell = \boldsymbol{X}^\top(\boldsymbol{y} - \hat{\boldsymbol{p}})$ — derived via two applications of the chain rule |
| Gradient ascent update | $\boldsymbol{w}_{t+1} = \boldsymbol{w}_t + \eta\boldsymbol{X}^\top(\boldsymbol{y}-\hat{\boldsymbol{p}}_t)$ |

---

**Module 02 complete.**

You can now compute partial derivatives of any order for the functions encountered in this course, state when mixed partials commute, construct tangent plane approximations, and apply the multivariable chain rule through computational graphs. The logistic regression gradient is no longer a formula to memorise — it is the chain rule applied twice, with the intermediate algebra written out.

**Up next**: Module 03 — The Gradient & Directional Derivatives

Module 02 gave us the tools to compute derivatives. Module 03 assembles them into the **gradient vector** $\nabla f$, proves that the gradient points perpendicular to level sets in the direction of steepest ascent (the claim we made in Module 01 without proof), defines the **directional derivative**, and introduces the **Jacobian** as the full derivative of a vector-valued function.

---
*Module 02, Unit 04 — Threat Surfaces: Multivariable Calculus for AI Security | fischer³ Education*